In [ ]:
'''
Feature set : B_LINEAGE_FREQ_ONLY
n_features  : 161
n_cat       : 22
n_num       : 139

Stage Q     : XGBoost, quality 6-class
             등외 / 3 / 2 / 1 / 1+ / 1++

Stage Y     : LightGBM, yield 3-class
             C / B / A
             단, 등외 제외 row만 학습

CV          : StratifiedKFold 3-fold, random_state=42
Weight      : balanced class weight ** 0.5

Final score : alpha * log(P_quality) + beta * log(P_yield)
alpha       : 1.15
beta        : 0.85

Class bias  : 고정 bias 사용
Strength    : 0.75
'''

In [ ]:
# ============================================================
# B_LINEAGE_FREQ_ONLY
# Hierarchical 3-fold model reproduction
# Final: class-bias strength 0.75
# Output:
# /content/drive/MyDrive/하늘/submissions/submission_B_hier3fold_classbias_s0p75_*.csv
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
import os
import gc
import json
import time
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb
import lightgbm as lgb
from packaging import version


# ============================================================
# 0. Config
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/하늘")

TRAIN_PATH = BASE_DIR / "model_feature_lineage_diagnostic_v1/train_B_lineage_freq_only.parquet"
TEST_PATH  = BASE_DIR / "model_feature_lineage_diagnostic_v1/test_B_lineage_freq_only.parquet"

RAW_TEST_PATH = Path("/content/drive/MyDrive/하늘/test_hanwoo.csv")

SUBMIT_DIR = BASE_DIR / "submissions"
OUT_DIR = BASE_DIR / "model_output_hier_clean_v1/B_hier3fold_classbias_s0p75_reproduce"

SUBMIT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "LAST_GRADE"
RANDOM_STATE = 42
N_SPLITS = 3

BEST_ALPHA = 1.15
BEST_BETA = 0.85
BIAS_STRENGTH = 0.75

print("xgboost version :", xgb.__version__)
print("lightgbm version:", lgb.__version__)
print("train path:", TRAIN_PATH)
print("test path :", TEST_PATH)
print("raw test  :", RAW_TEST_PATH)
print("out dir   :", OUT_DIR)


# ============================================================
# 1. Load data
# ============================================================

assert TRAIN_PATH.exists(), f"train parquet 없음: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"test parquet 없음: {TEST_PATH}"
assert RAW_TEST_PATH.exists(), f"raw test csv 없음: {RAW_TEST_PATH}"

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)

assert TARGET_COL in train_df.columns
assert TARGET_COL not in test_df.columns
assert len(train_df) == 2_408_699
assert len(test_df) == 452_497


# ============================================================
# 2. Feature columns
# ============================================================

raw_lineage_id_cols = [
    "CATTLE_NO",
    "FARM_UNIQUE_NO",
    "KPN_NO",
    "FATHER_CATTLE_NO",
    "MOTHER_ANIMAL_NO",
    "F_GMOTHER_ANIMAL_NO",
    "F_GFATHER_CATTLE_NO",
    "M_GMOTHER_ANIMAL_NO",
    "M_GFATHER_CATTLE_NO",
]

drop_cols = [TARGET_COL] + [c for c in raw_lineage_id_cols if c in train_df.columns]

feature_cols = [
    c for c in train_df.columns
    if c not in drop_cols and c in test_df.columns
]

CAT_CANDIDATES = [
    "sido",
    "sigungu",
    "JUDGE_SEX",
    "JUDGE_YEAR",
    "abatt_month",
    "season_code",
    "age_band",
    "weight_band",
    "density_band",
    "scale_band",
    "climate_cluster",
    "is_summer",
    "is_winter",
    "is_female",
    "is_male",
    "is_castrated",
    "sex_code",
    "scale_increased",
    "area_record_missing",
    "death_flag_before_abatt",
    "death_flag_365d_before_abatt",
    "death_flag_180d_before_abatt",
]

auto_cat_cols = [
    c for c in feature_cols
    if str(train_df[c].dtype) in ["object", "string", "category"]
]

categorical_cols = sorted(set(
    [c for c in CAT_CANDIDATES if c in feature_cols] + auto_cat_cols
))
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print("n_features    :", len(feature_cols))
print("n_categorical :", len(categorical_cols))
print("n_numeric     :", len(numeric_cols))
print("categorical_cols:")
print(categorical_cols)

assert len(feature_cols) == 161, f"feature 수가 다름. expected 161, got {len(feature_cols)}"


# ============================================================
# 3. Label decomposition
# ============================================================

GRADE_ORDER = [
    "등외",
    "3C", "3B", "3A",
    "2C", "2B", "2A",
    "1C", "1B", "1A",
    "1+C", "1+B", "1+A",
    "1++C", "1++B", "1++A",
]

QUALITY_ORDER = ["등외", "3", "2", "1", "1+", "1++"]
YIELD_ORDER_NONOUT = ["C", "B", "A"]

grade_to_id = {g: i for i, g in enumerate(GRADE_ORDER)}
id_to_grade = {i: g for i, g in enumerate(GRADE_ORDER)}

quality_to_id = {g: i for i, g in enumerate(QUALITY_ORDER)}
yield_to_id = {g: i for i, g in enumerate(YIELD_ORDER_NONOUT)}

def split_last_grade(label):
    label = str(label)

    if label == "등외":
        return "등외", "등외"

    if label.startswith("1++"):
        return "1++", label[-1]

    if label.startswith("1+"):
        return "1+", label[-1]

    if label.startswith("1"):
        return "1", label[-1]

    if label.startswith("2"):
        return "2", label[-1]

    if label.startswith("3"):
        return "3", label[-1]

    raise ValueError(f"Unknown label: {label}")

label_series = train_df[TARGET_COL].astype(str)

unknown = sorted(set(label_series.unique()) - set(GRADE_ORDER))
if len(unknown) > 0:
    raise ValueError(f"GRADE_ORDER에 없는 class: {unknown}")

y_16 = label_series.map(grade_to_id).astype(np.int16).values

quality_labels = []
yield_labels = []

for g in label_series:
    q, y = split_last_grade(g)
    quality_labels.append(q)
    yield_labels.append(y)

quality_labels = pd.Series(quality_labels)
yield_labels = pd.Series(yield_labels)

y_q = quality_labels.map(quality_to_id).astype(np.int16).values
nonout_mask = label_series.values != "등외"

print("y_16 shape:", y_16.shape)
print("y_q shape :", y_q.shape)
print("nonout rows:", int(nonout_mask.sum()))


# ============================================================
# 4. Preprocessing
#    categorical: train+test combined category code
#    numeric: train median fill
# ============================================================

def make_encoded_train_test(train_df, test_df, feature_cols, categorical_cols):
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()

    cat_cols = [c for c in categorical_cols if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    medians = {}

    for c in cat_cols:
        combined = pd.concat(
            [
                X_train[c].astype("string"),
                X_test[c].astype("string"),
            ],
            axis=0,
            ignore_index=True,
        ).fillna("MISSING")

        cats = pd.Index(combined.unique())

        X_train[c] = pd.Categorical(
            X_train[c].astype("string").fillna("MISSING"),
            categories=cats,
        ).codes.astype(np.int32)

        X_test[c] = pd.Categorical(
            X_test[c].astype("string").fillna("MISSING"),
            categories=cats,
        ).codes.astype(np.int32)

    for c in num_cols:
        tr = pd.to_numeric(X_train[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
        te = pd.to_numeric(X_test[c], errors="coerce").replace([np.inf, -np.inf], np.nan)

        med = tr.median()
        if pd.isna(med):
            med = 0.0

        medians[c] = float(med)

        X_train[c] = tr.fillna(med).astype(np.float32)
        X_test[c] = te.fillna(med).astype(np.float32)

    return X_train, X_test, cat_cols, num_cols, medians

start = time.time()

X_all, X_test, cat_cols, num_cols, medians = make_encoded_train_test(
    train_df=train_df,
    test_df=test_df,
    feature_cols=feature_cols,
    categorical_cols=categorical_cols,
)

print("X_all :", X_all.shape)
print("X_test:", X_test.shape)
print("preprocess elapsed min:", round((time.time() - start) / 60, 2))

assert X_all.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0


# ============================================================
# 5. Model utilities
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def make_soft_sample_weight(y_train, n_classes, alpha=0.5):
    classes = np.arange(n_classes)

    base_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train,
    )

    soft_weights = base_weights ** alpha
    weight_map = {cls: w for cls, w in zip(classes, soft_weights)}

    sample_weight = np.array(
        [weight_map[int(v)] for v in y_train],
        dtype=np.float32,
    )

    return sample_weight, base_weights, soft_weights

def get_xgb_params(num_class, random_state=42):
    xgb_version = version.parse(xgb.__version__)

    params = {
        "objective": "multi:softprob",
        "num_class": num_class,
        "eval_metric": "mlogloss",
        "n_estimators": 1400,
        "learning_rate": 0.045,
        "max_depth": 6,
        "min_child_weight": 30,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.1,
        "reg_lambda": 2.0,
        "max_bin": 256,
        "random_state": random_state,
        "n_jobs": 2,
        "verbosity": 1,
    }

    if xgb_version >= version.parse("2.0.0"):
        params.update({
            "tree_method": "hist",
            "device": "cuda",
        })
    else:
        params.update({
            "tree_method": "gpu_hist",
            "predictor": "gpu_predictor",
        })

    return params

def fit_xgb_multiclass(
    X_tr,
    y_tr,
    X_val,
    y_val,
    num_class,
    random_state=42,
    alpha=0.5,
    verbose=100,
):
    sample_weight, base_weights, soft_weights = make_soft_sample_weight(
        y_train=y_tr,
        n_classes=num_class,
        alpha=alpha,
    )

    params = get_xgb_params(
        num_class=num_class,
        random_state=random_state,
    )

    model = xgb.XGBClassifier(**params)

    try:
        model.fit(
            X_tr,
            y_tr,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            verbose=verbose,
        )
        device_used = "cuda"

    except Exception as e:
        print("[XGBoost GPU failed]")
        print("error:", repr(e))
        print("Fallback to CPU hist")

        params_cpu = params.copy()
        params_cpu.pop("device", None)
        params_cpu.pop("predictor", None)
        params_cpu["tree_method"] = "hist"
        params_cpu["n_jobs"] = -1

        model = xgb.XGBClassifier(**params_cpu)

        model.fit(
            X_tr,
            y_tr,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            verbose=verbose,
        )
        device_used = "cpu"

    return model, device_used, base_weights, soft_weights

def fit_lgbm_multiclass(
    X_tr,
    y_tr,
    X_val,
    y_val,
    num_class,
    categorical_features,
    random_state=42,
    alpha=0.5,
    verbose_eval=100,
):
    sample_weight, base_weights, soft_weights = make_soft_sample_weight(
        y_train=y_tr,
        n_classes=num_class,
        alpha=alpha,
    )

    cat_features_lgb = [c for c in categorical_features if c in X_tr.columns]

    params = {
        "objective": "multiclass",
        "num_class": num_class,
        "metric": "multi_logloss",
        "learning_rate": 0.045,
        "num_leaves": 63,
        "max_depth": -1,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.85,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "lambda_l1": 0.1,
        "lambda_l2": 1.0,
        "max_bin": 255,
        "max_cat_threshold": 64,
        "cat_l2": 10.0,
        "cat_smooth": 10.0,
        "verbosity": -1,
        "seed": random_state,
        "feature_pre_filter": False,
    }

    train_data = lgb.Dataset(
        X_tr,
        label=y_tr,
        weight=sample_weight,
        categorical_feature=cat_features_lgb,
        free_raw_data=False,
    )

    valid_data = lgb.Dataset(
        X_val,
        label=y_val,
        reference=train_data,
        categorical_feature=cat_features_lgb,
        free_raw_data=False,
    )

    model = lgb.train(
        params=params,
        train_set=train_data,
        num_boost_round=2500,
        valid_sets=[valid_data],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=150),
            lgb.log_evaluation(period=verbose_eval),
        ],
    )

    return model, base_weights, soft_weights


# ============================================================
# 6. 3-fold hierarchical training
#    Stage Q: XGBoost quality 6-class
#    Stage Y: LightGBM yield 3-class, 등외 제외
# ============================================================

seed_everything(RANDOM_STATE)

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

n_train = len(train_df)
n_test = len(test_df)

oof_q_proba = np.zeros((n_train, len(QUALITY_ORDER)), dtype=np.float32)
oof_y_proba = np.zeros((n_train, len(YIELD_ORDER_NONOUT)), dtype=np.float32)

test_q_proba_folds = []
test_y_proba_folds = []

fold_results = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all, y_16), start=1):
    print("\n" + "=" * 100)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 100)

    fold_start = time.time()

    X_tr = X_all.iloc[tr_idx]
    X_val = X_all.iloc[val_idx]

    yq_tr = y_q[tr_idx]
    yq_val = y_q[val_idx]

    # --------------------------------------------------------
    # Stage Q: quality 6-class
    # --------------------------------------------------------
    print("\n[Stage Q] XGBoost quality 6-class")

    q_model, q_device, _, _ = fit_xgb_multiclass(
        X_tr=X_tr,
        y_tr=yq_tr,
        X_val=X_val,
        y_val=yq_val,
        num_class=len(QUALITY_ORDER),
        random_state=RANDOM_STATE + fold,
        alpha=0.5,
        verbose=100,
    )

    q_val_proba = q_model.predict_proba(X_val).astype(np.float32)
    q_test_proba = q_model.predict_proba(X_test).astype(np.float32)

    oof_q_proba[val_idx] = q_val_proba
    test_q_proba_folds.append(q_test_proba)

    q_pred = q_val_proba.argmax(axis=1)
    q_macro = f1_score(yq_val, q_pred, average="macro", zero_division=0)
    q_acc = accuracy_score(yq_val, q_pred)

    print(f"[Stage Q] macro-F1={q_macro:.6f}, acc={q_acc:.6f}, device={q_device}")

    # --------------------------------------------------------
    # Stage Y: yield 3-class, non-out only
    # --------------------------------------------------------
    print("\n[Stage Y] LightGBM yield 3-class, non-out only")

    tr_nonout_idx = tr_idx[nonout_mask[tr_idx]]
    val_nonout_idx = val_idx[nonout_mask[val_idx]]

    Xy_tr = X_all.iloc[tr_nonout_idx]
    Xy_val = X_all.iloc[val_nonout_idx]

    yy_tr = yield_labels.iloc[tr_nonout_idx].map(yield_to_id).astype(np.int16).values
    yy_val = yield_labels.iloc[val_nonout_idx].map(yield_to_id).astype(np.int16).values

    y_model, _, _ = fit_lgbm_multiclass(
        X_tr=Xy_tr,
        y_tr=yy_tr,
        X_val=Xy_val,
        y_val=yy_val,
        num_class=len(YIELD_ORDER_NONOUT),
        categorical_features=cat_cols,
        random_state=RANDOM_STATE + 100 + fold,
        alpha=0.5,
        verbose_eval=100,
    )

    y_val_proba_nonout = y_model.predict(
        Xy_val,
        num_iteration=y_model.best_iteration,
    ).astype(np.float32)

    y_test_proba = y_model.predict(
        X_test,
        num_iteration=y_model.best_iteration,
    ).astype(np.float32)

    oof_y_proba[val_nonout_idx] = y_val_proba_nonout
    test_y_proba_folds.append(y_test_proba)

    y_pred = y_val_proba_nonout.argmax(axis=1)
    y_macro = f1_score(yy_val, y_pred, average="macro", zero_division=0)
    y_acc = accuracy_score(yy_val, y_pred)

    print(f"[Stage Y] macro-F1={y_macro:.6f}, acc={y_acc:.6f}")

    fold_elapsed = (time.time() - fold_start) / 60

    fold_results.append({
        "fold": fold,
        "q_macro_f1": float(q_macro),
        "q_accuracy": float(q_acc),
        "q_device": q_device,
        "y_macro_f1": float(y_macro),
        "y_accuracy": float(y_acc),
        "elapsed_minutes": float(fold_elapsed),
        "n_train": int(len(tr_idx)),
        "n_valid": int(len(val_idx)),
        "n_train_nonout_y": int(len(tr_nonout_idx)),
        "n_valid_nonout_y": int(len(val_nonout_idx)),
    })

    pd.DataFrame(fold_results).to_csv(
        OUT_DIR / "fold_stage_results_running.csv",
        index=False,
        encoding="utf-8-sig",
    )

    del X_tr, X_val, Xy_tr, Xy_val
    del q_model, y_model
    del q_val_proba, q_test_proba, y_val_proba_nonout, y_test_proba
    gc.collect()

print("\n========== 2-stage OOF training done ==========")

fold_results_df = pd.DataFrame(fold_results)
fold_results_df.to_csv(
    OUT_DIR / "fold_stage_results_final.csv",
    index=False,
    encoding="utf-8-sig",
)
print(fold_results_df)


# ============================================================
# 7. Average test probabilities
# ============================================================

test_q_proba = np.mean(test_q_proba_folds, axis=0).astype(np.float32)
test_y_proba = np.mean(test_y_proba_folds, axis=0).astype(np.float32)

np.save(OUT_DIR / "oof_q_proba.npy", oof_q_proba)
np.save(OUT_DIR / "oof_y_proba.npy", oof_y_proba)
np.save(OUT_DIR / "test_q_proba.npy", test_q_proba)
np.save(OUT_DIR / "test_y_proba.npy", test_y_proba)
np.save(OUT_DIR / "y_16.npy", y_16)

print("saved probabilities:", OUT_DIR)


# ============================================================
# 8. Hierarchical 16-class scoring
# ============================================================

EPS = 1e-12

grade_quality_id = []
grade_yield_id = []

for g in GRADE_ORDER:
    q, y = split_last_grade(g)
    grade_quality_id.append(quality_to_id[q])

    if g == "등외":
        grade_yield_id.append(-1)
    else:
        grade_yield_id.append(yield_to_id[y])

grade_quality_id = np.array(grade_quality_id, dtype=np.int16)
grade_yield_id = np.array(grade_yield_id, dtype=np.int16)

def normalize_proba(p):
    p = np.asarray(p, dtype=np.float32)
    p = np.clip(p, EPS, 1.0)
    return p / p.sum(axis=1, keepdims=True)

def make_hier_scores(q_proba, y_proba, alpha=1.15, beta=0.85):
    q_proba = normalize_proba(q_proba)
    y_proba = normalize_proba(y_proba)

    n = q_proba.shape[0]
    scores = np.zeros((n, len(GRADE_ORDER)), dtype=np.float32)

    log_q = np.log(np.clip(q_proba, EPS, 1.0))
    log_y = np.log(np.clip(y_proba, EPS, 1.0))

    for k, g in enumerate(GRADE_ORDER):
        qid = grade_quality_id[k]
        yid = grade_yield_id[k]

        if g == "등외":
            scores[:, k] = alpha * log_q[:, qid]
        else:
            scores[:, k] = alpha * log_q[:, qid] + beta * log_y[:, yid]

    return scores

oof_scores = make_hier_scores(
    oof_q_proba,
    oof_y_proba,
    alpha=BEST_ALPHA,
    beta=BEST_BETA,
)

test_scores = make_hier_scores(
    test_q_proba,
    test_y_proba,
    alpha=BEST_ALPHA,
    beta=BEST_BETA,
)

base_oof_pred = oof_scores.argmax(axis=1)

base_macro = f1_score(y_16, base_oof_pred, average="macro", zero_division=0)
base_acc = accuracy_score(y_16, base_oof_pred)
base_bal = balanced_accuracy_score(y_16, base_oof_pred)

print("\n[Base hierarchical OOF]")
print("macro_f1         :", base_macro)
print("accuracy         :", base_acc)
print("balanced_accuracy:", base_bal)


# ============================================================
# 9. Fixed class-bias s0p75
#    이 값이 submission_B_hier3fold_classbias_s0p75 계열 핵심
# ============================================================

FIXED_PUBLIC_BEST_BIAS = np.array([
    -2.00,  # 등외
    -0.30,  # 3C
    -0.55,  # 3B
    -0.40,  # 3A
     0.30,  # 2C
     0.00,  # 2B
    -0.10,  # 2A
     0.30,  # 1C
     0.10,  # 1B
     0.10,  # 1A
     0.20,  # 1+C
     0.10,  # 1+B
     0.10,  # 1+A
    -0.20,  # 1++C
    -0.20,  # 1++B
    -0.10,  # 1++A
], dtype=np.float32)

class_bias = FIXED_PUBLIC_BEST_BIAS.copy()
applied_bias = class_bias * BIAS_STRENGTH

final_oof_scores = oof_scores + applied_bias.reshape(1, -1)
final_oof_pred = final_oof_scores.argmax(axis=1)

final_macro = f1_score(y_16, final_oof_pred, average="macro", zero_division=0)
final_acc = accuracy_score(y_16, final_oof_pred)
final_bal = balanced_accuracy_score(y_16, final_oof_pred)

print("\n[Final OOF after class-bias s0p75]")
print("bias_strength    :", BIAS_STRENGTH)
print("macro_f1         :", final_macro)
print("accuracy         :", final_acc)
print("balanced_accuracy:", final_bal)

bias_df = pd.DataFrame({
    "LAST_GRADE": GRADE_ORDER,
    "class_bias": class_bias,
    "applied_bias": applied_bias,
})

print(bias_df)

bias_df.to_csv(
    OUT_DIR / "class_bias_s0p75.csv",
    index=False,
    encoding="utf-8-sig",
)

report_df = pd.DataFrame(
    classification_report(
        y_16,
        final_oof_pred,
        labels=list(range(len(GRADE_ORDER))),
        target_names=GRADE_ORDER,
        output_dict=True,
        zero_division=0,
    )
).T

report_df.to_csv(
    OUT_DIR / "oof_report_s0p75.csv",
    encoding="utf-8-sig",
)


# ============================================================
# 10. Test prediction
# ============================================================

adjusted_test_scores = test_scores + applied_bias.reshape(1, -1)

test_pred_id = adjusted_test_scores.argmax(axis=1).astype(np.int16)
test_pred_label = np.array([id_to_grade[int(i)] for i in test_pred_id])

pred_dist = (
    pd.Series(test_pred_label, name=TARGET_COL)
    .value_counts()
    .reindex(GRADE_ORDER)
    .fillna(0)
    .astype(int)
    .reset_index()
)

pred_dist.columns = [TARGET_COL, "count"]
pred_dist["ratio"] = pred_dist["count"] / pred_dist["count"].sum()

print("\n[Test prediction distribution]")
print(pred_dist)

np.save(OUT_DIR / "test_pred_id_s0p75.npy", test_pred_id)


# ============================================================
# 11. Make final submission
# ============================================================

raw_test = pd.read_csv(RAW_TEST_PATH, encoding="utf-8-sig")

print("raw_test:", raw_test.shape)

assert len(raw_test) == len(test_pred_label)
assert len(raw_test) == 452_497

submission = raw_test.copy()
submission[TARGET_COL] = test_pred_label

cols = [c for c in submission.columns if c != TARGET_COL] + [TARGET_COL]
submission = submission[cols]

invalid = sorted(set(submission[TARGET_COL].astype(str)) - set(GRADE_ORDER))
assert len(invalid) == 0, invalid
assert submission[TARGET_COL].isna().sum() == 0

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

submit_path = SUBMIT_DIR / f"submission_B_hier3fold_classbias_s0p75_{timestamp}.csv"
dist_path = SUBMIT_DIR / f"pred_dist_B_hier3fold_classbias_s0p75_{timestamp}.csv"

submission.to_csv(submit_path, index=False, encoding="utf-8-sig")
pred_dist.to_csv(dist_path, index=False, encoding="utf-8-sig")

summary = {
    "feature_set": "B_LINEAGE_FREQ_ONLY",
    "train_path": str(TRAIN_PATH),
    "test_path": str(TEST_PATH),
    "raw_test_path": str(RAW_TEST_PATH),
    "n_features": len(feature_cols),
    "n_categorical": len(categorical_cols),
    "n_numeric": len(numeric_cols),
    "n_splits": N_SPLITS,
    "random_state": RANDOM_STATE,
    "stage_q_model": "XGBoost multi:softprob quality 6-class",
    "stage_y_model": "LightGBM multiclass yield 3-class non-out only",
    "best_alpha": BEST_ALPHA,
    "best_beta": BEST_BETA,
    "bias_strength": BIAS_STRENGTH,
    "base_oof_macro_f1": float(base_macro),
    "base_oof_accuracy": float(base_acc),
    "base_oof_balanced_accuracy": float(base_bal),
    "final_oof_macro_f1_s0p75": float(final_macro),
    "final_oof_accuracy_s0p75": float(final_acc),
    "final_oof_balanced_accuracy_s0p75": float(final_bal),
    "submission_path": str(submit_path),
    "pred_dist_path": str(dist_path),
}

with open(OUT_DIR / "run_summary_s0p75.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDONE")
print("saved submission:", submit_path)
print("saved pred_dist :", dist_path)
print("saved summary   :", OUT_DIR / "run_summary_s0p75.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
xgboost version : 3.2.0
lightgbm version: 4.6.0
train path: /content/drive/MyDrive/하늘/model_feature_lineage_diagnostic_v1/train_B_lineage_freq_only.parquet
test path : /content/drive/MyDrive/하늘/model_feature_lineage_diagnostic_v1/test_B_lineage_freq_only.parquet
raw test  : /content/drive/MyDrive/하늘/test_hanwoo.csv
out dir   : /content/drive/MyDrive/하늘/model_output_hier_clean_v1/B_hier3fold_classbias_s0p75_reproduce
train_df: (2408699, 162)
test_df : (452497, 161)
n_features    : 161
n_categorical : 22
n_numeric     : 139
categorical_cols:
['JUDGE_SEX', 'JUDGE_YEAR', 'abatt_month', 'age_band', 'area_record_missing', 'climate_cluster', 'death_flag_180d_before_abatt', 'death_flag_365d_before_abatt', 'death_flag_before_abatt', 'density_band', 'is_castrated', 'is_female', 'is_male', 'is_summer', 'is_winter', 'scale_band', 'scale_increased', 'season_code', 'sex